# Chain Rule and Backpropagation

Companion notebook for the [Chain Rule and Backpropagation](https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/02-chain-rule-and-backpropagation) lesson.

We implement backpropagation from scratch and compare to numerical gradients.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Chain rule: step-by-step example

For $h(x) = \sigma(3x^2)$, backpropagate from $dL/dh = 1$.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
def sigmoid_grad(z): return sigmoid(z) * (1 - sigmoid(z))

x = 0.5   # input

# Forward pass (build computational graph)
g = 3 * x**2        # g = 3x²
h = sigmoid(g)      # h = σ(g)

print('=== Forward Pass ===')
print(f'x = {x}')
print(f'g = 3x² = {g}')
print(f'h = σ(g) = {h:.6f}')

# Backward pass
dL_dh = 1.0             # upstream gradient
dL_dg = dL_dh * sigmoid_grad(g)  # local gradient of σ
dL_dx = dL_dg * 6 * x            # local gradient of 3x²

print('\n=== Backward Pass ===')
print(f'dL/dh = {dL_dh}')
print(f'dL/dg = dL/dh × σ\'(g) = {dL_dh:.4f} × {sigmoid_grad(g):.4f} = {dL_dg:.6f}')
print(f'dL/dx = dL/dg × 6x   = {dL_dg:.6f} × {6*x:.4f} = {dL_dx:.6f}')

# Verify numerically
eps = 1e-7
h_plus  = sigmoid(3*(x+eps)**2)
h_minus = sigmoid(3*(x-eps)**2)
numerical = (h_plus - h_minus) / (2 * eps)
print(f'\nNumerical dh/dx = {numerical:.6f}  (matches: {abs(dL_dx - numerical) < 1e-5})')

## Manual backprop through a 1-hidden-layer network

Same recipe, scaled up: a $3 \to 4$ (ReLU) $\to 1$ (sigmoid) network with binary
cross-entropy loss. We do the forward pass, run backprop **by hand** (one line per
node, upstream x local), then verify *every* gradient against a central-difference
numerical gradient.

In [ ]:
def relu(z): return np.maximum(0, z)
def relu_grad(z): return (z > 0).astype(float)

rng = np.random.default_rng(0)

# Network: 3 inputs -> 4 hidden (ReLU) -> 1 output (sigmoid)
x  = np.array([1.0, 2.0, 3.0])
W1 = rng.standard_normal((4, 3)) * 0.1
b1 = np.zeros(4)
w2 = rng.standard_normal(4) * 0.1
b2 = 0.0
y  = 1.0  # true label


def forward(W1, b1, w2, b2):
    """Forward pass returning the scalar loss (used for both training and FD checks)."""
    z1    = W1 @ x + b1
    a1    = relu(z1)
    z2    = w2 @ a1 + b2
    y_hat = sigmoid(z2)
    L     = -(y * np.log(y_hat + 1e-12) + (1 - y) * np.log(1 - y_hat + 1e-12))
    return L, (z1, a1, z2, y_hat)


L, (z1, a1, z2, y_hat) = forward(W1, b1, w2, b2)

# Backward pass: one line per node, upstream x local
dL_dz2 = y_hat - y                 # BCE + sigmoid collapse to (y_hat - y)
dL_dw2 = dL_dz2 * a1               # x a1
dL_db2 = dL_dz2                     # x 1
dL_da1 = dL_dz2 * w2               # x w2
dL_dz1 = dL_da1 * relu_grad(z1)    # x ReLU'(z1)
dL_dW1 = np.outer(dL_dz1, x)       # x x^T
dL_db1 = dL_dz1                     # x 1

print('Loss: {0:.6f}'.format(L))
print('Prediction: {0:.4f}'.format(y_hat))
print('dL/dw2 (shape {0}): {1}'.format(dL_dw2.shape, dL_dw2.round(6)))
print('dL/dW1 (shape {0}):\n{1}'.format(dL_dW1.shape, dL_dW1.round(6)))


# --- Finite-difference check of every analytic gradient ---
def numerical_grad(param_name, base):
    """Central-difference gradient of L w.r.t. a copy of one parameter array/scalar."""
    eps = 1e-6
    arr = np.atleast_1d(np.array(base, dtype=float))
    grad = np.zeros_like(arr)
    for idx in np.ndindex(arr.shape):
        plus = arr.copy();  plus[idx]  += eps
        minus = arr.copy(); minus[idx] -= eps
        kwargs_p = {'W1': W1, 'b1': b1, 'w2': w2, 'b2': b2}
        kwargs_m = dict(kwargs_p)
        kwargs_p[param_name] = plus.reshape(np.shape(base))
        kwargs_m[param_name] = minus.reshape(np.shape(base))
        Lp, _ = forward(**kwargs_p)
        Lm, _ = forward(**kwargs_m)
        grad[idx] = (Lp - Lm) / (2 * eps)
    return grad.reshape(np.shape(base))


checks = {
    'W1': (dL_dW1, numerical_grad('W1', W1)),
    'b1': (dL_db1, numerical_grad('b1', b1)),
    'w2': (dL_dw2, numerical_grad('w2', w2)),
    'b2': (np.array(dL_db2), numerical_grad('b2', b2)),
}

print('\n=== Gradient check (analytic vs numerical) ===')
for name, (analytic, numeric) in checks.items():
    max_err = np.max(np.abs(np.asarray(analytic) - np.asarray(numeric)))
    print('{0:>3}: max abs error = {1:.2e}  (ok: {2})'.format(name, max_err, max_err < 1e-5))

## Manual backprop through a 1-hidden-layer network

In [ ]:
def relu(z): return np.maximum(0, z)
def relu_grad(z): return (z > 0).astype(float)

rng = np.random.default_rng(0)

# Network: 3 inputs → 4 hidden (ReLU) → 1 output (sigmoid)
x  = np.array([1.0, 2.0, 3.0])
W1 = rng.standard_normal((4, 3)) * 0.1
b1 = np.zeros(4)
w2 = rng.standard_normal(4) * 0.1
b2 = 0.0
y  = 1.0  # true label

# Forward pass
z1    = W1 @ x + b1
a1    = relu(z1)
z2    = w2 @ a1 + b2
y_hat = sigmoid(z2)
L     = -(y * np.log(y_hat + 1e-12) + (1-y) * np.log(1-y_hat + 1e-12))

# Backward pass
dL_dz2 = y_hat - y
dL_dw2 = dL_dz2 * a1
dL_db2 = dL_dz2
dL_da1 = dL_dz2 * w2
dL_dz1 = dL_da1 * relu_grad(z1)
dL_dW1 = np.outer(dL_dz1, x)
dL_db1 = dL_dz1

print(f'Loss: {L:.6f}')
print(f'Prediction: {y_hat:.4f}')
print(f'dL/dw2 (shape {dL_dw2.shape}): {dL_dw2.round(6)}')
print(f'dL/dW1 (shape {dL_dW1.shape}): {dL_dW1.round(6)}')

## Vanishing gradients: sigmoid vs ReLU

In [ ]:
def simulate_gradient_flow(activation, activation_grad, n_layers=20, x0=1.0):
    """Simulate gradient magnitude through n_layers with a given activation."""
    rng = np.random.default_rng(42)
    gradient = 1.0
    grad_magnitudes = [gradient]

    for _ in range(n_layers):
        z = rng.standard_normal() * 0.5   # random pre-activation
        local_grad = activation_grad(z)
        gradient *= local_grad
        grad_magnitudes.append(abs(gradient))

    return grad_magnitudes

sigmoid_grads = simulate_gradient_flow(sigmoid, sigmoid_grad, n_layers=20)
relu_grads    = simulate_gradient_flow(relu, relu_grad, n_layers=20)

fig, ax = plt.subplots(figsize=(10, 5))
layers = range(len(sigmoid_grads))
ax.semilogy(layers, sigmoid_grads, 'o-', color='#f97316', lw=2, ms=6, label='Sigmoid')
ax.semilogy(layers, relu_grads,    's-', color='#6366f1', lw=2, ms=6, label='ReLU')
ax.set_xlabel('Layer (counting backward)'); ax.set_ylabel('|gradient| (log scale)')
ax.set_title('Gradient magnitude through 20 layers: Sigmoid vs ReLU', pad=12)
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

print(f'Sigmoid gradient after 20 layers: {sigmoid_grads[-1]:.2e}')
print(f'ReLU    gradient after 20 layers: {relu_grads[-1]:.2e}')

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Chain rule by hand

Take the single-neuron function $f(w) = \sigma(wx + b)$ and differentiate it **with respect to the weight** $w$. Two links in the chain:

$$\frac{\partial f}{\partial w} = \underbrace{\sigma'(z)}_{\text{outer}} \cdot \underbrace{\frac{\partial z}{\partial w}}_{= \, x}
\qquad \text{with } z = wx + b, \;\; \sigma'(z) = \sigma(z)\,(1 - \sigma(z))$$

The checks compare your formula against a numerical gradient — the same gradient-checking trick used to debug real backprop code.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def grad_w(x, w, b):
    """d sigmoid(w*x + b) / dw via the chain rule."""
    # TODO(you): the pre-activation z
    z = ...

    # TODO(you): sigma(z), then sigma'(z) = s * (1 - s)
    s = ...

    # TODO(you): chain rule: sigma'(z) times dz/dw (which is x)
    return ...

In [ ]:
# Checks — run me
x, w, b = 2.0, 0.5, -1.0
h = 1e-6
numeric = (sigmoid((w + h) * x + b) - sigmoid((w - h) * x + b)) / (2 * h)
assert abs(grad_w(x, w, b) - numeric) < 1e-8, "chain rule must match the numerical gradient"
assert abs(grad_w(0.0, w, b)) < 1e-12, "x = 0 -> z doesn't depend on w -> zero gradient"
assert abs(grad_w(1.0, 0.0, 0.0) - 0.25) < 1e-12, "sigma'(0) = 0.25, times x = 1"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def grad_w(x, w, b):
    z = w * x + b
    s = sigmoid(z)
    return s * (1 - s) * x
```

</details>

### Exercise 2 — Backprop through a tiny network

A two-weight scalar network: $y = w_2 \cdot \text{ReLU}(w_1 x)$, loss $L = \tfrac{1}{2}(y - t)^2$. Walk the gradient backwards node by node:

$$\frac{\partial L}{\partial y} = y - t, \qquad
\frac{\partial L}{\partial w_2} = (y - t)\,a, \qquad
\frac{\partial L}{\partial w_1} = (y - t)\,w_2 \cdot \mathbb{1}[z > 0] \cdot x$$

where $z = w_1 x$ and $a = \text{ReLU}(z)$. The last two checks make the **dead ReLU** point: when $z < 0$, the gate is shut and *neither* weight gets a gradient.

In [ ]:
def backprop(x, w1, w2, t):
    """Return (dL/dw1, dL/dw2) for y = w2 * relu(w1 * x), L = 0.5 * (y - t)^2."""
    # Forward pass
    z = w1 * x
    a = max(z, 0.0)          # ReLU
    y = w2 * a

    # Backward pass
    # TODO(you): dL/dy
    dy = ...

    # TODO(you): dL/dw2 = dL/dy * dy/dw2 (= a)
    dw2 = ...

    # TODO(you): dL/dw1 = dL/dy * w2 * relu'(z) * x, where relu'(z) is 1 if z > 0 else 0
    dw1 = ...

    return dw1, dw2

In [ ]:
# Checks — run me
def loss(x, w1, w2, t):
    return 0.5 * (w2 * max(w1 * x, 0.0) - t) ** 2

x, w1, w2, t = 1.5, 0.8, -1.2, 1.0
dw1, dw2 = backprop(x, w1, w2, t)
h = 1e-6
num1 = (loss(x, w1 + h, w2, t) - loss(x, w1 - h, w2, t)) / (2 * h)
num2 = (loss(x, w1, w2 + h, t) - loss(x, w1, w2 - h, t)) / (2 * h)
assert abs(dw1 - num1) < 1e-6, "dL/dw1 must match the numerical gradient"
assert abs(dw2 - num2) < 1e-6, "dL/dw2 must match the numerical gradient"

dw1_dead, dw2_dead = backprop(-1.5, 0.8, -1.2, 1.0)
assert dw1_dead == 0.0, "ReLU is dead (z < 0) -> gradient to w1 is blocked"
assert dw2_dead == 0.0, "dead ReLU -> a = 0 -> w2 gets no gradient either"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def backprop(x, w1, w2, t):
    z = w1 * x
    a = max(z, 0.0)
    y = w2 * a
    dy = y - t
    dw2 = dy * a
    dw1 = dy * w2 * (x if z > 0 else 0.0)
    return dw1, dw2
```

</details>